# GPU runbook: Qwen3.5 on a benchmark manifest

Runs on **Google Colab** with an **L4** (Runtime → Change runtime type → L4; A100 also works). **Qwen3.5 failed to start on a T4** in our attempt (vLLM 0.32, engine init error on compute capability 7.5, 30 Oct 2026). Free Colab only offers a T4, so this needs paid compute units; Colab's *Resources* panel shows the L4's hourly rate.

**What it does:** starts a vLLM server with Qwen3.5 on this GPU, installs `qdu` with uv, downloads the datasets, runs a smoke test on the largest pages and then the paths in `PIPELINES` on `MANIFEST`, against `localhost` (so measured latency has no network time). Predictions are written to **Google Drive** after every call, so a disconnect loses nothing: run the cells again and finished calls are skipped.

**Full labelling (`benchmarks/full.csv`, 12,305 questions) in two stages under the same `RUN_NAME`:**
1. `PIPELINES = "vision"` needs no OCR, so it can start while `ocr_prepass.ipynb` fills the OCR cache on a CPU runtime. 12,305 calls.
2. `PIPELINES = "text,hybrid"` once the pre-pass has finished (cell 7 checks). 24,610 calls.

Calls run one at a time, so each call's latency is a clean per-question cost. In the pilot (4 MP image cap) a vision call took 1.2 s and a text call 0.25 s on average; with the model-native cap the largest pages cost more. Expect very roughly 6–10 h for stage 1 and 8–12 h for stage 2; the progress lines show the real pace. Colab sessions end after at most ~24 h (less on some plans), so plan on resuming.

When you are done: **Runtime → Disconnect and delete runtime**, otherwise the session keeps using compute units.

In [ ]:
# 1. Settings
import os
import subprocess

MODEL = "Qwen/Qwen3.5-4B"
RUN_NAME = "full-qwen3.5-4b"  # never reuse a finished paid run's name for a different setup
MANIFEST = "benchmarks/full.csv"  # check run: benchmarks/check_52.csv; pilot: benchmarks/pilot_300.csv
SMOKE = "benchmarks/smoke_largest.csv"  # one question on each of the 15 largest pages
PIPELINES = "vision"  # full.csv stage 1: "vision", stage 2: "text,hybrid"; check run: "text,vision,hybrid"
OCR = "paddle-mobile"  # must match ocr_prepass.ipynb
DRIVE = "/content/drive/MyDrive/qdu"
OCR_CACHE = f"{DRIVE}/ocr_cache"
RUNS_DIR = f"{DRIVE}/runs"
REPO = "https://github.com/emre-geris/query-based-document-understanding.git"
# Private repo? Put a GitHub token in Colab Secrets as GITHUB_TOKEN.

SPEC = f"openai:{MODEL}"
NEEDS_OCR = bool({"text", "hybrid"} & set(PIPELINES.split(",")))
cap = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
).stdout.strip()
print(cap)
major = float(cap.splitlines()[0].split(",")[1])
DTYPE = "bfloat16" if major >= 8 else "float16"  # T4 (7.5) has no bf16
print("dtype:", DTYPE, "| pipelines:", PIPELINES, "| needs OCR:", NEEDS_OCR)

In [ ]:
# 2. Drive
# Google Drive keeps the OCR cache and the runs across sessions: a Colab runtime loses its disk
# when it disconnects, and the full run spans several sessions.
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(OCR_CACHE, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
# 3. System packages: uv and vLLM (takes a few minutes)
!pip -q install -U uv vllm
!uv --version && python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 4. Start the vLLM server, detached from the notebook (setsid), so interrupting a cell
#    never kills it. First start downloads the model (~9 GB) and compiles kernels (~7 min on L4).
#    If this cell times out, just run it again: it does not start a second server.
#
#    Prefix and multimodal caches are OFF on purpose: a cache hit on a page sent before would
#    make the later call look cheaper than it is. Latency is a reported cost, so every call
#    must pay its full price.
#    --max-model-len 32768: pages are sent up to Qwen3.5's own 16.8 MP limit (~16k image
#    tokens) plus OCR text in the hybrid path.
#    The flags are exported as QDU_SERVER_ARGS so every run records them in run_info.json.
import time
import urllib.request

VLLM_FLAGS = (
    f"--port 8000 --dtype {DTYPE} --max-model-len 32768 --gpu-memory-utilization 0.90 "
    "--limit-mm-per-prompt '{\"image\": 1}' "
    "--no-enable-prefix-caching --mm-processor-cache-gb 0"
)
os.environ["QDU_SERVER_ARGS"] = f"vllm serve {MODEL} {VLLM_FLAGS}"


def server_ready() -> bool:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        return True
    except Exception:
        return False


running = subprocess.run(["pgrep", "-f", "[v]llm serve"], capture_output=True).returncode == 0
if not running:
    cmd = f"setsid nohup vllm serve {MODEL} {VLLM_FLAGS} > /content/vllm.log 2>&1 < /dev/null &"
    print(cmd)
    subprocess.run(cmd, shell=True)
start = time.time()
while not server_ready() and time.time() - start < 30 * 60:
    time.sleep(30)
    last = subprocess.run(["tail", "-n", "1", "/content/vllm.log"], capture_output=True, text=True)
    print(f"{int(time.time() - start):>5} s | {last.stdout.strip()[:150]}")
print("server ready" if server_ready() else "server not ready yet: check vllm.log or re-run this cell")

In [ ]:
# 5. Code, installed with uv (PaddleOCR only when the text/hybrid paths run)
token = os.environ.get("GITHUB_TOKEN")
try:
    from google.colab import userdata

    token = token or userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
os.chdir("/content")
if not os.path.isdir("qdu-repo"):
    subprocess.run(["git", "clone", "-q", url, "qdu-repo"], check=True)
os.chdir("/content/qdu-repo")
!git pull -q && git log --oneline -1
EXTRA = "--extra paddle" if NEEDS_OCR else ""
!uv sync -q {EXTRA} && uv run qdu --help | head -3

In [ ]:
# 6. Datasets -> HF cache (~1.6 GB, all four)
!uv run qdu data download

In [ ]:
# 7. Text/hybrid only: OCR before any model call. For a small manifest (a check run) the pages
#    are OCR'd here, on this VM's CPU, while the GPU is idle; for full.csv run the pre-pass
#    notebook instead and this only checks that the cache is complete. Either way every row
#    records the CPU its OCR time was measured on.
if NEEDS_OCR and MANIFEST != "benchmarks/full.csv":
    !uv run qdu ocr warm --engine {OCR} --manifest {MANIFEST} --ocr-cache {OCR_CACHE}
if NEEDS_OCR:
    !uv run python -c "from pathlib import Path; from qdu.data import read_manifest; from qdu.ocr import create_ocr; from qdu.ocr.base import cache_file; e = create_ocr('{OCR}', Path('{OCR_CACHE}')); docs = {{r.doc_id for r in read_manifest(Path('{MANIFEST}'))}}; missing = [d for d in docs if not cache_file(e.cache_dir, d).exists()]; print(len(docs) - len(missing), '/', len(docs), 'pages cached;', len(missing), 'missing')"

In [ ]:
# 8. Smoke test on the 15 largest pages: no errors, short answers (not reasoning), and the
#    largest input fits the context. Its latencies are not part of the results.
!uv run qdu benchmark run --text-model {SPEC} --vision-model {SPEC} --pipelines {PIPELINES} --ocr {OCR} --ocr-cache {OCR_CACHE} --manifest {SMOKE} --runs-dir {RUNS_DIR} --name smoke-{RUN_NAME}
!uv run python -c "import json; [print(r['pipeline'], '|', r['prediction'], '| gold', r['answers'], '| tok', r['input_tokens'], r['output_tokens'], '| s', round(r['model_latency_s'] or 0, 2), r['error'] or '') for r in map(json.loads, open('{RUNS_DIR}/smoke-{RUN_NAME}/predictions.jsonl'))]"

In [ ]:
# 9. The run: MANIFEST x PIPELINES. Resumable: after a disconnect run cells 1-6 again, then this.
!uv run qdu benchmark run --text-model {SPEC} --vision-model {SPEC} --pipelines {PIPELINES} --ocr {OCR} --ocr-cache {OCR_CACHE} --manifest {MANIFEST} --runs-dir {RUNS_DIR} --name {RUN_NAME}

In [ ]:
# 10. After both stages: report and package the run (it is also on Drive)
!uv run qdu benchmark report {RUNS_DIR}/{RUN_NAME}
!cd {RUNS_DIR} && zip -qr /content/{RUN_NAME}.zip {RUN_NAME} && ls -lh /content/{RUN_NAME}.zip